# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre Moreira Barros

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# ETAPA 2 — BIG DATA: EXECUÇÃO DO PIPELINE

## 1. Configuração inicial

In [39]:
# 1. CONFIGURAÇÃO

!pip -q install duckdb pyspark pyarrow fastparquet plotly

import os
import shutil
import pandas as pd
import numpy as np
import plotly.express as px
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Diretório principal do projeto
BASE = "./content/bigdata/"

# Deixa o notebook reproduzível:
# remove uma execução anterior, caso exista.
if os.path.exists(BASE):
    shutil.rmtree(BASE)

print("✓ Ambiente inicializado")
print(f"✓ Diretório de trabalho: {BASE}")


✓ Ambiente inicializado
✓ Diretório de trabalho: ./content/bigdata/


## 2. Upload do  dataset

In [40]:
# 2. GERAR DATASET

#!/usr/bin/env python3

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

N = 30_000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

print("Gerando base de dados da avaliação...")

# ----------------------------------------------------------------------
# Segmento e score (mesma lógica conceitual do curso, valores diferentes)
# ----------------------------------------------------------------------
segments = np.random.choice(['Premium', 'Standard', 'High-Risk'], size=N, p=[0.55, 0.35, 0.10])
credit_score = np.clip(np.random.normal(640, 140, N).astype(int), 300, 900)

# ----------------------------------------------------------------------
# Novas dimensões — não existiam no dataset do curso
# ----------------------------------------------------------------------
channel = np.random.choice(['app', 'web', 'pos', 'atm'], size=N, p=[0.40, 0.30, 0.20, 0.10])
merchant_category = np.random.choice(
    ['varejo', 'viagem', 'eletronico', 'alimentacao', 'servicos', 'saude'],
    size=N, p=[0.30, 0.10, 0.15, 0.20, 0.15, 0.10]
)
transaction_type = np.random.choice(['compra', 'saque', 'transferencia', 'pagamento'], size=N, p=[0.55, 0.15, 0.20, 0.10])

# ----------------------------------------------------------------------
# Valores e datas
# ----------------------------------------------------------------------
amount = np.clip(np.random.lognormal(4.3, 1.3, N), 5, 40_000)
dates = [START_DATE + timedelta(days=random.randint(0, (END_DATE-START_DATE).days),
                                  hours=random.randint(0, 23)) for _ in range(N)]
risk_score = np.random.uniform(0, 100, N)

# ----------------------------------------------------------------------
# Fraude: NOVA lógica — depende de channel + merchant_category + hora
# (padrão real: fraude em apps costuma concentrar em compras de viagem
#  feitas de madrugada — o aluno precisa DESCOBRIR isso na EDA)
# ----------------------------------------------------------------------
is_fraud = []
for i in range(N):
    base = {'Premium': 0.004, 'Standard': 0.012, 'High-Risk': 0.045}[segments[i]]
    if channel[i] == 'app':
        base *= 2.2
    if merchant_category[i] == 'viagem':
        base *= 2.8
    if dates[i].hour < 5:  # madrugada
        base *= 1.8
    if risk_score[i] > 80:
        base *= 1.5
    is_fraud.append(random.random() < min(base, 0.35))

status = ['declined' if f or random.random() < 0.08 else 'approved' for f in is_fraud]

df = pd.DataFrame({
    'transaction_id': range(1, N+1),
    'customer_id': np.random.randint(1, 8000, N),
    'amount': amount.round(2),
    'transaction_type': transaction_type,
    'channel': channel,
    'merchant_category': merchant_category,
    'timestamp': dates,
    'status': status,
    'risk_score': risk_score.round(2),
    'segment': segments,
    'credit_score': credit_score,
    'is_fraud': is_fraud,
})
df = df.sort_values('timestamp').reset_index(drop=True)

df.to_csv('./avaliacao_transactions.csv', index=False)

print(f"✓ {len(df)} linhas geradas")
print(f"  Taxa de fraude geral: {df['is_fraud'].mean():.2%}")
print(f"  Fraude por channel:")
print(df.groupby('channel')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))
print(f"  Fraude por merchant_category:")
print(df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))
print(f"  Fraude por segmento:")
print(df.groupby('segment')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))


Gerando base de dados da avaliação...
✓ 30000 linhas geradas
  Taxa de fraude geral: 2.50%
  Fraude por channel:
channel
app    3.79%
atm    1.80%
web    1.74%
pos    1.42%
Name: is_fraud, dtype: object
  Fraude por merchant_category:
merchant_category
viagem         5.32%
saude          2.47%
alimentacao    2.31%
varejo         2.24%
servicos       1.97%
eletronico     1.96%
Name: is_fraud, dtype: object
  Fraude por segmento:
segment
High-Risk    9.67%
Standard     2.93%
Premium      0.95%
Name: is_fraud, dtype: object


## 3. Criar a estrutura de pastas

In [41]:
# Estrutura de dados da avaliação TechPay:
# RAW → BRONZE → SILVER → GOLD

base_path = "./content/bigdata"

folders = [
    f"{base_path}/raw",
    f"{base_path}/bronze",
    f"{base_path}/silver",
    f"{base_path}/gold",
    f"{base_path}/evidencias"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Estrutura criada.")

Estrutura criada.


## 4. Mover Dataset para raw

In [42]:
# A avaliação fornece uma única base contendo as transações
# e seus respectivos atributos, incluindo o rótulo de fraude.

import shutil

shutil.move(
    "./avaliacao_transactions.csv",
    f"{base_path}/raw/avaliacao_transactions.csv"
)

print("Arquivo movido para RAW.")

Arquivo movido para RAW.


In [43]:
raw_file = f"{base_path}/raw/avaliacao_transactions.csv"

print(os.path.exists(raw_file))
print(os.path.getsize(raw_file), "bytes")

True
2772012 bytes


## 5. Inspecionar Dataset

In [44]:
# Realiza uma inspeção rápida do dataset da camada 'raw',
# exibindo informações básicas e uma amostra inicial

label = "transações"

print("=" * 70)

# Imprime o nome do dataset em maiúsculas
print(label.upper())

# Carrega o CSV da camada RAW
df_inspect = pd.read_csv(
    f"{base_path}/raw/avaliacao_transactions.csv"
)

# Imprime o número de linhas e colunas
print(
    f"Linhas: {len(df_inspect):,} | "
    f"Colunas: {len(df_inspect.columns)}"
)

# Imprime os nomes das colunas
print("Colunas:", ", ".join(df_inspect.columns))
print()

# Exibe as 3 primeiras linhas do DataFrame
display(df_inspect.head(3))

TRANSAÇÕES
Linhas: 30,000 | Colunas: 12
Colunas: transaction_id, customer_id, amount, transaction_type, channel, merchant_category, timestamp, status, risk_score, segment, credit_score, is_fraud



,transaction_id,customer_id,amount,transaction_type,channel,merchant_category,timestamp,status,risk_score,segment,credit_score,is_fraud
0,25607,4169,160.41,transferencia,web,eletronico,2025-01-01 00:00:00,approved,26.71,High-Risk,577,False
1,3433,4220,21.12,compra,app,servicos,2025-01-01 00:00:00,approved,57.70,Premium,612,False
2,20910,7081,94.62,compra,app,viagem,2025-01-01 00:00:00,approved,42.39,Premium,810,False


In [45]:
pip show pyspark

Name: pyspark
Version: 4.2.0
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: Apache-2.0
Location: C:\Users\aj_ol\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages
Requires: py4j
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 6. Inicializar o PySpark

In [46]:
# Agora vamos utilizar o Spark local.

spark = (
    SparkSession.builder
    .appName("TechPay-BigData")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark

In [47]:
print(spark.version)

4.2.0


## 7. Etapa 1 — Ingestão

In [48]:
# Verificar o arquivo com DuckDB:

import duckdb

con = duckdb.connect()

df_check = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{raw_file}')
    LIMIT 5
""").df()

df_check

,transaction_id,customer_id,amount,transaction_type,channel,merchant_category,timestamp,status,risk_score,segment,credit_score,is_fraud
0,25607,4169,160.41,transferencia,web,eletronico,2025-01-01 00:00:00,approved,26.71,High-Risk,577,False
1,3433,4220,21.12,compra,app,servicos,2025-01-01 00:00:00,approved,57.70,Premium,612,False
2,20910,7081,94.62,compra,app,viagem,2025-01-01 00:00:00,approved,42.39,Premium,810,False
3,15864,81,224.00,compra,web,viagem,2025-01-01 01:00:00,approved,58.75,Premium,589,False
4,16181,215,801.25,compra,app,varejo,2025-01-01 01:00:00,approved,23.29,Premium,618,False


In [49]:
# Contagem de linhas na RAW

count_raw = con.execute(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{raw_file}')
""").fetchone()[0]

print("Quantidade de registros RAW:", count_raw)

Quantidade de registros RAW: 30000


## 8. Criar o schema correto

In [50]:
# Definir Schema

schema = StructType([
    StructField("transaction_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("segment", StringType(), True),
    StructField("credit_score", IntegerType(), True),
    StructField("is_fraud", BooleanType(), True)
])

df_raw = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(raw_file)
)

df_raw.show(5, truncate=False)

+--------------+-----------+------+----------------+-------+-----------------+-------------------+--------+----------+---------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|timestamp          |status  |risk_score|segment  |credit_score|is_fraud|
+--------------+-----------+------+----------------+-------+-----------------+-------------------+--------+----------+---------+------------+--------+
|25607         |4169       |160.41|transferencia   |web    |eletronico       |2025-01-01 00:00:00|approved|26.71     |High-Risk|577         |false   |
|3433          |4220       |21.12 |compra          |app    |servicos         |2025-01-01 00:00:00|approved|57.7      |Premium  |612         |false   |
|20910         |7081       |94.62 |compra          |app    |viagem           |2025-01-01 00:00:00|approved|42.39     |Premium  |810         |false   |
|15864         |81         |224.0 |compra          |web    |viagem           |2025-01-01 01:00

## 9. Validar a RAW


In [51]:
# Verificar quantidade
print("RAW:", df_raw.count())

RAW: 30000


In [52]:
# Checar duplicatas de transaction_id

duplicates = (
    df_raw.groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicates.show()

print("IDs duplicados:", duplicates.count())

+--------------+-----+
|transaction_id|count|
+--------------+-----+
+--------------+-----+

IDs duplicados: 0


In [53]:
# Checar valores nulos

nulls = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])

nulls.show()

+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|timestamp|status|risk_score|segment|credit_score|is_fraud|
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
|             0|          0|     0|               0|      0|                0|        0|     0|         0|      0|           0|       0|
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+



In [54]:
# Checar valores negativos

df_raw.filter(F.col("amount") < 0).show()

+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|timestamp|status|risk_score|segment|credit_score|is_fraud|
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+



In [55]:
# Checar Risk Score inválido

df_raw.filter(
    (F.col("risk_score") < 0) |
    (F.col("risk_score") > 100)
).show()

# Checar Credit Score inválido

df_raw.filter(
    (F.col("credit_score") < 300) |
    (F.col("credit_score") > 900)
).show()

+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|timestamp|status|risk_score|segment|credit_score|is_fraud|
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+

+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|timestamp|status|risk_score|segment|credit_score|is_fraud|
+--------------+-----------+------+----------------+-------+-----------------+---------+------+----------+-------+------------+--------+
+--------------+-----------+------+-----

In [56]:
# Checar Status
df_raw.groupBy("status").count().show()

# Checar Channel
df_raw.groupBy("channel").count().show()

# Checar Fraud
df_raw.groupBy("is_fraud").count().show()

+--------+-----+
|  status|count|
+--------+-----+
|approved|26948|
|declined| 3052|
+--------+-----+

+-------+-----+
|channel|count|
+-------+-----+
|    web| 9004|
|    app|12021|
|    pos| 5968|
|    atm| 3007|
+-------+-----+

+--------+-----+
|is_fraud|count|
+--------+-----+
|   false|29249|
|    true|  751|
+--------+-----+



## 10. Etapa 2 — Particionamento

Aqui iremos particionar por ano e mês da transação.

In [57]:
df_raw_partitioned = (
    df_raw
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
)

## 11. Salvar RAW em Parquet

In [58]:
# raw_parquet = f"{base_path}/raw/parquet/"
# print (f"Salvando RAW particionado em: {raw_parquet}")
# (
#     df_raw_partitioned
#     .write
#     .mode("overwrite")
#     .partitionBy("year", "month")
#     .parquet(raw_parquet)
# )

# !find f"{base_path}/raw/parquet" -maxdepth 2 -type d

import duckdb
from pathlib import Path

raw_parquet = Path(base_path) / "raw" / "parquet"

print(f"Salvando RAW particionado em: {raw_parquet}")

# Garante que a pasta exista
raw_parquet.mkdir(parents=True, exist_ok=True)

# Converte o DataFrame Spark para Pandas
df_raw_pd = df_raw_partitioned.toPandas()

# Grava como Parquet particionado usando DuckDB
con = duckdb.connect()

con.register("df_raw", df_raw_pd)

con.execute(f"""
    COPY df_raw
    TO '{raw_parquet.as_posix()}'
    (
        FORMAT PARQUET,
        PARTITION_BY (year, month),
        OVERWRITE_OR_IGNORE
    )
""")

con.close()

print("✓ RAW particionado salvo com sucesso!")

from pathlib import Path

for path in raw_parquet.rglob("*"):
    if path.is_dir():
        print(path)


Salvando RAW particionado em: content\bigdata\raw\parquet
✓ RAW particionado salvo com sucesso!
content\bigdata\raw\parquet\year=2025
content\bigdata\raw\parquet\year=2025\month=1
content\bigdata\raw\parquet\year=2025\month=10
content\bigdata\raw\parquet\year=2025\month=11
content\bigdata\raw\parquet\year=2025\month=12
content\bigdata\raw\parquet\year=2025\month=2
content\bigdata\raw\parquet\year=2025\month=3
content\bigdata\raw\parquet\year=2025\month=4
content\bigdata\raw\parquet\year=2025\month=5
content\bigdata\raw\parquet\year=2025\month=6
content\bigdata\raw\parquet\year=2025\month=7
content\bigdata\raw\parquet\year=2025\month=8
content\bigdata\raw\parquet\year=2025\month=9


## 12. Etapa 3 — Bronze

Agora começa efetivamente a higienização da base.

A camada Bronze não deve transformar completamente os dados.

Ela deve:

* Remover duplicidades;
* Normalizar textos;
* Eliminar registros claramente inválidos;
* Manter estrutura próxima da origem;
* Registrar/permitir rastreabilidade.

In [59]:
df_bronze = df_raw_partitioned

#### Normalizar campos textuais

In [60]:
text_columns = [
    "transaction_type",
    "channel",
    "merchant_category",
    "status",
    "segment"
]

for c in text_columns:
    df_bronze = df_bronze.withColumn(
        c,
        F.lower(F.trim(F.col(c)))
    )

#### Remover duplicidades

In [61]:
before_dedup = df_bronze.count()

df_bronze = df_bronze.dropDuplicates(["transaction_id"])

after_dedup = df_bronze.count()

print("Antes:", before_dedup)
print("Depois:", after_dedup)
print("Duplicatas removidas:", before_dedup - after_dedup)

Antes: 30000
Depois: 30000
Duplicatas removidas: 0


#### Remover valores impossíveis

In [62]:
# Valor
df_bronze = df_bronze.filter(
    F.col("amount").isNotNull() &
    (F.col("amount") >= 0)
)

# Risk Score
df_bronze = df_bronze.filter(
    F.col("risk_score").isNotNull() &
    (F.col("risk_score") >= 0) &
    (F.col("risk_score") <= 100)
)

# Credit Score
df_bronze = df_bronze.filter(
    F.col("credit_score").isNotNull() &
    (F.col("credit_score") >= 300) &
    (F.col("credit_score") <= 900)
)

#Campos fundamentais
df_bronze = df_bronze.filter(
    F.col("transaction_id").isNotNull() &
    F.col("customer_id").isNotNull() &
    F.col("timestamp").isNotNull()
)

#### Validação Final da Bronze

In [63]:
bronze_count = df_bronze.count()

print("Registros Bronze:", bronze_count)
print("Registros descartados:", count_raw - bronze_count)

Registros Bronze: 30000
Registros descartados: 0


#### Salvar Bronze

In [64]:
# bronze_path = f"{base_path}/bronze"

# (
#     df_bronze
#     .write
#     .mode("overwrite")
#     .partitionBy("year", "month")
#     .parquet(bronze_path)
# )

import duckdb
from pathlib import Path

bronze_path = Path(base_path) / "bronze"

print(f"Salvando BRONZE particionado em: {bronze_path}")

# Garante que a pasta exista
bronze_path.mkdir(parents=True, exist_ok=True)

# Converte o DataFrame Spark para Pandas
df_bronze_pd = df_bronze.toPandas()

# Grava como Parquet particionado usando DuckDB
con = duckdb.connect()

con.register("df_bronze", df_bronze_pd)

con.execute(f"""
    COPY df_bronze
    TO '{bronze_path.as_posix()}'
    (
        FORMAT PARQUET,
        PARTITION_BY (year, month),
        OVERWRITE_OR_IGNORE
    )
""")

con.close()

print("✓ BRONZE particionado salvo com sucesso!")

Salvando BRONZE particionado em: content\bigdata\bronze
✓ BRONZE particionado salvo com sucesso!


## 14. Etapa 4 — Silver

Na camada Silver, deve-se enriquecer os dados.

Iremos manter channel e merchant_category pois será importante para responder perguntas como:

Qual canal apresenta maior fraude?
Qual categoria de estabelecimento concentra maior risco?
Qual canal movimenta mais dinheiro?
Qual categoria apresenta mais transações recusadas?

#### Criar período do dia

In [65]:
df_silver = (
    df_bronze
    .withColumn(
        "hour",
        F.hour("timestamp")
    )
    .withColumn(
        "date",
        F.to_date("timestamp")
    )
)

# Classificação
df_silver = df_silver.withColumn(
    "period",
    F.when((F.col("hour") >= 6) & (F.col("hour") < 12), "manha")
     .when((F.col("hour") >= 12) & (F.col("hour") < 18), "tarde")
     .when((F.col("hour") >= 18) & (F.col("hour") < 24), "noite")
     .otherwise("madrugada")
)


#### Criar faixa de valor

In [66]:
df_silver = df_silver.withColumn(
    "amount_range",
    F.when(F.col("amount") < 50, "ate_50")
     .when(F.col("amount") < 200, "50_200")
     .when(F.col("amount") < 500, "200_500")
     .otherwise("acima_500")
)

#### Criar faixa de risco

In [67]:
df_silver = df_silver.withColumn(
    "risk_level",
    F.when(F.col("risk_score") < 30, "baixo")
     .when(F.col("risk_score") < 70, "medio")
     .otherwise("alto")
)

#### Criar indicador de transação aprovada

In [68]:
# Indicador de transação aprovada


#### Validação Final da Silver

In [69]:
df_silver.printSchema()

df_silver.show(5, truncate=False)

print("Bronze:", bronze_count)
print("Silver:", df_silver.count())


root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- segment: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- period: string (nullable = false)
 |-- amount_range: string (nullable = false)
 |-- risk_level: string (nullable = false)

+--------------+-----------+-------+----------------+-------+-----------------+-------------------+--------+----------+--------+------------+--------+----+-----+----+----------+------+------------+----------+
|tra

#### Salvar Silver

In [70]:
# silver_path = f"{base_path}/etapa2_bigdata/silver"

# (
#     df_silver
#     .write
#     .mode("overwrite")
#     .partitionBy("year", "month")
#     .parquet(silver_path)
# )

silver_path = Path(base_path) / "silver"

print(f"Salvando SILVER particionado em: {silver_path}")

# Garante que a pasta exista
silver_path.mkdir(parents=True, exist_ok=True)

# Converte o DataFrame Spark para Pandas
df_silver_pd = df_silver.toPandas()

# Grava como Parquet particionado usando DuckDB
con = duckdb.connect()

con.register("df_silver", df_silver_pd)

con.execute(f"""
    COPY df_silver
    TO '{silver_path.as_posix()}'
    (
        FORMAT PARQUET,
        PARTITION_BY (year, month),
        OVERWRITE_OR_IGNORE
    )
""")

con.close()

print("✓ SILVER particionado salvo com sucesso!")

Salvando SILVER particionado em: content\bigdata\silver
✓ SILVER particionado salvo com sucesso!


## 15. Etapa 5 — Gold

Iremos desenvolver 4 tabelas agregadas diferentes para enriquecer as análises no dashboard.

Assim, teremos:


1.   Risco por Canal: para que possamos responder quais canais apresentam maior exposição a fraude e risco.
2.   Risco por Categoria: para identificar quais categorias de estabelecimento concentram maior risco.
3. Evolução temporal: para enriquecer o dashboard com análises históricas.
4. Risco de fraude: para responder onde estão as transações potencialmente mais perigosas.


#### GOLD 1 — Risco por canal

In [71]:
gold_channel = (
    df_silver
    .withColumn(
        "fraud_amount",
        F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)
    )
    .withColumn(
        "is_declined",
        F.when(F.col("status") == 'declined', True).otherwise(False)
    )
    .groupBy("channel")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.sum(F.col("is_fraud").cast("integer")).alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount"),
        F.sum(F.col("is_declined").cast("integer")).alias("declined_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .withColumn(
        "declined_rate",
        F.col("declined_transactions") / F.col("total_transactions")
    )
)

gold_channel.show()

+-------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+--------------------+-------------------+
|channel|total_transactions|      total_amount|        avg_amount|    avg_risk_score|fraud_transactions|      fraud_amount|declined_transactions|          fraud_rate|      declined_rate|
+-------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+--------------------+-------------------+
|    web|              9004|1573259.1999999958| 174.7289204797863| 50.36443913816094|               157|          31579.22|                  853|0.017436694802310086|0.09473567303420702|
|    app|             12021|2007907.0599999952|167.03328009316988| 49.82269694700933|               455| 63909.90999999995|                 1333|0.037850428416937025|0.11088927709841111|
|    pos|              5968|1088597.6699999978| 182.4057758042892

#### GOLD 2 — Risco por categoria

In [72]:
gold_category = (
    df_silver
    .withColumn(
        "fraud_amount",
        F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)
    )
    .withColumn(
        "is_declined",
        F.when(F.col("status") == 'declined', True).otherwise(False)
    )
    .groupBy("merchant_category")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.sum(F.col("is_fraud").cast("integer")).alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount"),
        F.sum(F.col("is_declined").cast("integer")).alias("declined_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .withColumn(
        "declined_rate",
        F.col("declined_transactions") / F.col("total_transactions")
    )
)

gold_category.show()

+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+--------------------+-------------------+
|merchant_category|total_transactions|      total_amount|        avg_amount|    avg_risk_score|fraud_transactions|      fraud_amount|declined_transactions|          fraud_rate|      declined_rate|
+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+--------------------+-------------------+
|      alimentacao|              5976|1055465.6499999992| 176.6174113119142|49.685634203480475|               138|21017.000000000004|                  585|0.023092369477911646|0.09789156626506024|
|           viagem|              3010| 545957.3099999997|181.38116611295672| 50.82484717607971|               160|25966.400000000012|                  365|0.053156146179401995| 0.1212624584717608|
|           var

#### GOLD 3 — Evolução temporal

In [73]:
gold_daily = (
    df_silver
    .withColumn(
        "fraud_amount",
        F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)
    )
    .withColumn(
        "is_declined",
        F.when(F.col("status") == 'declined', True).otherwise(False)
    )
    .groupBy("date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.sum(F.col("is_fraud").cast("integer")).alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount"),
        F.sum(F.col("is_declined").cast("integer")).alias("declined_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
)
gold_daily.show()

+----------+------------------+------------------+------------------+------------------+------------+---------------------+--------------------+
|      date|total_transactions|      total_amount|    avg_risk_score|fraud_transactions|fraud_amount|declined_transactions|          fraud_rate|
+----------+------------------+------------------+------------------+------------------+------------+---------------------+--------------------+
|2025-10-15|                95|14037.589999999998|59.010210526315774|                 1|      152.56|                   10|0.010526315789473684|
|2025-05-30|                86|11665.699999999995| 52.17116279069768|                 1|       21.25|                    9|0.011627906976744186|
|2025-10-21|                76|15700.869999999999| 40.80736842105264|                 1|       83.29|                    8|0.013157894736842105|
|2025-11-08|                73| 9395.289999999997| 50.22684931506847|                 2|      337.32|                   11|  0.027

#### GOLD 4 — Risco de Fraude

In [74]:
gold_risk = (
    df_silver
    .withColumn(
        "fraud_amount",
        F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)
    )
    .groupBy("risk_level")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.sum(F.col("is_fraud").cast("integer")).alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
)
gold_risk.show()

+----------+------------------+------------------+------------------+------------------+------------------+--------------------+
|risk_level|total_transactions|      total_amount|    avg_risk_score|fraud_transactions|      fraud_amount|          fraud_rate|
+----------+------------------+------------------+------------------+------------------+------------------+--------------------+
|     medio|             11874|2074461.7200000088| 50.00684689236999|               263|33547.600000000006|0.022149233619673237|
|      alto|              9091| 1547624.109999998| 84.95472885271157|               276|          47557.97| 0.03035969640303597|
|     baixo|              9035|1585757.6700000006|15.173275041505258|               212| 38262.35999999997|0.023464305478693968|
+----------+------------------+------------------+------------------+------------------+------------------+--------------------+



#### Salvar tabelas Gold

In [75]:
gold_path = Path(base_path) / "gold"

print(f"Salvando tabelas GOLD em: {gold_path}")

# Garante que a pasta exista
gold_path.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()


# ============================================================
# GOLD — RISCO POR CANAL
# ============================================================

gold_channel_pd = gold_channel.toPandas()

con.register("gold_channel", gold_channel_pd)

con.execute(f"""
    COPY gold_channel
    TO '{(gold_path / "risk_by_channel.parquet").as_posix()}'
    (
        FORMAT PARQUET,
        OVERWRITE_OR_IGNORE
    )
""")

print("✓ Gold — risk_by_channel salvo")


# ============================================================
# GOLD — RISCO POR CATEGORIA
# ============================================================

gold_category_pd = gold_category.toPandas()

con.register("gold_category", gold_category_pd)

con.execute(f"""
    COPY gold_category
    TO '{(gold_path / "risk_by_category.parquet").as_posix()}'
    (
        FORMAT PARQUET,
        OVERWRITE_OR_IGNORE
    )
""")

print("✓ Gold — risk_by_category salvo")


# ============================================================
# GOLD — RISCO DIÁRIO
# ============================================================

gold_daily_pd = gold_daily.toPandas()

con.register("gold_daily", gold_daily_pd)

con.execute(f"""
    COPY gold_daily
    TO '{(gold_path / "daily_risk.parquet").as_posix()}'
    (
        FORMAT PARQUET,
        OVERWRITE_OR_IGNORE
    )
""")

print("✓ Gold — daily_risk salvo")


# ============================================================
# GOLD — NÍVEL DE RISCO
# ============================================================

gold_risk_pd = gold_risk.toPandas()

con.register("gold_risk", gold_risk_pd)

con.execute(f"""
    COPY gold_risk
    TO '{(gold_path / "risk_level.parquet").as_posix()}'
    (
        FORMAT PARQUET,
        OVERWRITE_OR_IGNORE
    )
""")

print("✓ Gold — risk_level salvo")

print("\n✓ Todas as tabelas GOLD foram salvas com sucesso!")


Salvando tabelas GOLD em: content\bigdata\gold
✓ Gold — risk_by_channel salvo
✓ Gold — risk_by_category salvo
✓ Gold — daily_risk salvo
✓ Gold — risk_level salvo

✓ Todas as tabelas GOLD foram salvas com sucesso!


#### Testar a Gold com DuckDB

In [76]:
result = con.execute(f"""
    SELECT *
    FROM read_parquet('{(gold_path / "risk_by_channel.parquet").as_posix()}')
    ORDER BY fraud_rate DESC
""").df()

con.close()

result

,channel,total_transactions,total_amount,avg_amount,avg_risk_score,fraud_transactions,fraud_amount,declined_transactions,fraud_rate,declined_rate
0,app,12021,2007907.06,167.033280,49.822697,455,63909.91,1333,0.037850,0.110889
1,atm,3007,538079.57,178.942325,49.502441,54,12395.30,308,0.017958,0.102428
2,web,9004,1573259.20,174.728920,50.364439,157,31579.22,853,0.017437,0.094736
3,pos,5968,1088597.67,182.405776,50.593396,85,11483.50,558,0.014243,0.093499


## 16. Validação final das tabelas

In [77]:
print("Bronze:", df_bronze.count())
print("Silver:", df_silver.count())
print("Gold canal:", gold_channel.count())
print("Gold categoria:", gold_category.count())
print("Gold diário:", gold_daily.count())
print("Gold risco:", gold_risk.count())

Bronze: 30000
Silver: 30000
Gold canal: 4
Gold categoria: 6
Gold diário: 365
Gold risco: 3


In [78]:
# Evidências

# Evidência 1 — Arquivo RAW
print("Evidência 1 — Arquivo RAW")
print(raw_file)
print("Tamanho:", os.path.getsize(raw_file), "bytes")

# Evidência 2 — Schema
print("Evidência 2 — Schema")
df_raw.printSchema()

# Evidência 3 — Dados iniciais
print("Evidência 3 — Dados iniciais")
df_raw.show(5)

# Evidência 4 — Problemas encontrados
print("Evidência 4 — Problemas encontrados")
print("Duplicatas:", duplicates.count())

print("Valores negativos:")
df_raw.filter(F.col("amount") < 0).show()

print("Risk score inválido:")
df_raw.filter(
    (F.col("risk_score") < 0) |
    (F.col("risk_score") > 100)
).show()

# Evidência 5 — Bronze
print("Evidência 5 — Bronze")
print("RAW:", df_raw.count())
print("BRONZE:", df_bronze.count())
print("Descartados:", df_raw.count() - df_bronze.count())

# Evidência 6 — Silver
print("Evidência 6 — Silver")
df_silver.select(
    "transaction_id",
    "channel",
    "merchant_category",
    "hour",
    "period",
    "amount_range",
    "risk_level"
).show(10)

# Evidência 7 — Gold
print("Evidência 7 — Gold")
gold_channel.show()
gold_category.show()

# Evidência 8 — estrutura Parquet
print("Evidência 8 — estrutura Parquet")
!find /content/avaliacao_final/etapa2_bigdata -type f | head -30

Evidência 1 — Arquivo RAW
./content/bigdata/raw/avaliacao_transactions.csv
Tamanho: 2772012 bytes
Evidência 2 — Schema
root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- segment: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)

Evidência 3 — Dados iniciais
+--------------+-----------+------+----------------+-------+-----------------+-------------------+--------+----------+---------+------------+--------+
|transaction_id|customer_id|amount|transaction_type|channel|merchant_category|          timestamp|  status|risk_score|  segment|credit_score|is_fraud|
+--------------+-----------+------+-

FIND: op��o inv�lida
'head' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [79]:
print('=== Schemas das Tabelas Gold ===')

print('\n--- gold_channel ---')
gold_channel.printSchema()

print('\n--- gold_category ---')
gold_category.printSchema()

print('\n--- gold_daily ---')
gold_daily.printSchema()

print('\n--- gold_risk ---')
gold_risk.printSchema()

=== Schemas das Tabelas Gold ===

--- gold_channel ---
root
 |-- channel: string (nullable = true)
 |-- total_transactions: long (nullable = false)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- avg_risk_score: double (nullable = true)
 |-- fraud_transactions: long (nullable = true)
 |-- fraud_amount: double (nullable = true)
 |-- declined_transactions: long (nullable = true)
 |-- fraud_rate: double (nullable = true)
 |-- declined_rate: double (nullable = true)


--- gold_category ---
root
 |-- merchant_category: string (nullable = true)
 |-- total_transactions: long (nullable = false)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- avg_risk_score: double (nullable = true)
 |-- fraud_transactions: long (nullable = true)
 |-- fraud_amount: double (nullable = true)
 |-- declined_transactions: long (nullable = true)
 |-- fraud_rate: double (nullable = true)
 |-- declined_rate: double (nullable = true)

